# Deep Learning in Audio Applications
### Teach machines to listen: your own audio-to-text notebook

You will take speech, turn it into text, see where that breaks, build the two ideas that let a
computer read a sequence (an **RNN**, then an **LSTM**), and end with **meeting minutes**. Run the
cells from top to bottom. Cells marked **YOUR TURN** have one line for you to write; a check
tells you straight away if it is right.

<details><summary>More detail: restarting, and what's in each section</summary>

> **If Colab restarts, or you jump ahead to a later section:** run cell 2 (Setup) again first, then
> every cell above the one you want, in order. Several cells further down reuse things an earlier
> cell set up (`transcript`, `turns`, the trained model) and will show a `NameError` without them.

| section | content | agenda block(s) | minutes |
|---|---|---|---|
| 1 | Audio to text with Python | 2 (part) | ~6 |
| 2 | Where transcription struggles | 2 (part) | ~4 |
| 3 | RNNs: reading words as a sequence (+ Hands-on A, + why memory fades) | 3, 4, 5 | 15 + 10 + 7 |
| 4 | LSTMs: an eraser you can set (+ teaching it, + Hands-on B) | 6, 7, 8 | 20 + 10 + 20 |
| 5 | Meeting minutes (+ your own meeting) | 9 | 15 |

</details>

## 0. Setup (run once, about a minute)
On Colab: **Runtime > Change runtime type > T4 GPU** makes Whisper several times faster (the free CPU works too, just slower).
Upload `workshop_files.zip` from the facilitators to the Files panel on the left before running this cell.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")                  # avoids a library clash on some Windows installs
import warnings; warnings.filterwarnings("ignore", message="FP16")  # Whisper on CPU only

if importlib.util.find_spec("whisper") is None:                       # Whisper: speech-to-text from OpenAI
    print("Installing Whisper, about a minute...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("workshop_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("workshop_files.zip").extractall(".")              # the code and audio for this workshop
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)   # ../prototype when run from the repo
PROTO or sys.exit("Upload workshop_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, PROTO)
print("ffmpeg (reads audio files):", "found" if shutil.which("ffmpeg") else "NOT FOUND - install it (Colab has it already)")
print("ready")

## 1. Audio to text with Python
*Facilitator-run, during the talk -- sections 1-2 together are Agenda block 2, about 10 minutes*
*(roughly 6 min here, 4 min in section 2). Participants: run these cells too (about a minute on*
*a T4); the later sections reuse their results. Nothing in either section is YOUR TURN.*

Whisper is a ready-made speech-to-text model. Three lines: load it, give it a file, read the text.
Bigger models hear more but run slower: `tiny.en`, `base.en`, `small.en`, `medium.en` (the `.en` ones know only English). We pick a size that fits your machine.


In [ ]:
import torch, whisper
import live_mic   # imported here, not gated behind block 2b (marked "optional")

MODEL = "small.en" if torch.cuda.is_available() else "base.en"        # a GPU can afford the bigger model
print("using", MODEL, "on", "the GPU" if torch.cuda.is_available() else "the CPU", "(about 20s on CPU)")
model = whisper.load_model(MODEL)
result = model.transcribe("audio/sample_meeting.wav", language="en")
print(result["text"])


### What Whisper is reading
A spectrogram is like sheet music for speech: **time runs left to right, pitch runs bottom to top**.
Here is the waveform and the spectrogram for a short clip: three sentences from the meeting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wave_samples = whisper.load_audio("audio/sample_short.wav")
t = np.arange(len(wave_samples)) / 16000

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(t, wave_samples, linewidth=0.5)
ax1.set_ylabel("amplitude")
ax1.set_title("Waveform")
with np.errstate(divide="ignore"):      # a silent stretch gives log10(0); matplotlib handles it, just don't warn
    ax2.specgram(wave_samples, Fs=16000, cmap="magma")
ax2.set_ylabel("pitch (Hz)")
ax2.set_xlabel("time (s)")
ax2.set_title("Spectrogram -- time left to right, pitch bottom to top")
plt.tight_layout()
plt.show()


**Order matters.** Play a short clip forwards, then backwards -- same sounds, but backwards it
stops being words. Whisper looks at the spectrogram in time order too -- it's a different kind
of model (it weighs a whole 30-second window at once, not one step at a time like the RNN
below), but order still matters to it: reverse the clip and the "words" it would read make no
sense either.


In [ ]:
from IPython.display import Audio, display

clip = wave_samples[:8000]          # about half a second -- roughly one word
print("forwards:")
display(Audio(clip, rate=16000))
print("backwards:")
display(Audio(clip[::-1], rate=16000))


**What Whisper-style models actually see.** Not the waveform directly -- a short window every
10 ms, turned into a handful of numbers (an MFCC frame, the classic speech-model feature; Whisper
itself uses a similar but larger spectrogram feature). This is the picture above, in the form a
model reads.


In [ ]:
from audio_features import mfcc

samples = whisper.load_audio("audio/sample_short.wav")
frames = mfcc(samples)
print(f"{len(samples) / 16000:.0f} seconds of audio  ->  {frames.shape[0]} frames x {frames.shape[1]} numbers (one frame every 10 ms)")


Our pipeline wants more than one block of text: it wants **who said what**. `get_transcript` listens for pauses and
starts a new turn each time one person stops and the next starts (here: a pause longer than 1.5 s).

In [ ]:
os.environ["WHISPER_MODEL"] = MODEL          # the pipeline's transcriber reads this
from transcriber import get_transcript

print("transcribing again with speaker splitting (about 20s on CPU)...")
transcript = get_transcript("audio/sample_meeting.wav", pause_gap=1.5)
print(transcript)


## 2. Where transcription struggles
*Facilitator-run, about 4 of block 2's 10 minutes (see section 1's note).*

Real recordings are not clean. The three things that hurt most: **background noise**, **a poor microphone**, and **accents**.
We score by how many of the words Whisper gets right (`word_accuracy`: 1.0 = every word).


In [ ]:
import numpy as np
from audio_check import word_accuracy

SAID = ("Venue is confirmed for the 14th, catering still pending. "
        "Can you follow up with catering by Friday? Yes, I'll email them today.")
clean = whisper.load_audio("audio/sample_short.wav")          # the same three sentences, 14 seconds

def hear(samples):
    return model.transcribe(samples, language="en")["text"].strip()

print("clean      ", f"{word_accuracy(SAID, hear(clean)):.0%}", "|", hear(clean), "  (about 5s on CPU)")


**Noise.** Add random hiss to the recording. How loud can it get before Whisper breaks? (try 0.02, 0.05, 0.1, 0.2)

In [ ]:
NOISE = 0.1
noisy = (clean + NOISE * np.random.default_rng(0).normal(size=clean.shape)).astype("float32")
heard = hear(noisy)
print(f"noise {NOISE}:", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**A poor microphone.** Keep only every 4th sample, like a cheap mic or a phone line. The high sounds (s, t, f) disappear.

In [ ]:
poor = np.repeat(clean[::4], 4).astype("float32")
heard = hear(poor)
print("poor mic  :", f"{word_accuracy(SAID, heard):.0%}", "|", heard)

**Accents.** Record yourself on your phone reading the same three sentences, with about 2 seconds between them:

> Venue is confirmed for the 14th, catering still pending. / Can you follow up with catering by Friday? / Yes, I'll email them today.

Upload it to the Files panel, put its name below, and compare with the clean voice. What did Whisper get wrong, and why?

In [ ]:
MY_VOICE = None            # for example "my_voice.m4a"
if MY_VOICE:
    mine = whisper.load_audio(MY_VOICE)
    heard = hear(mine)
    print("my voice  :", f"{word_accuracy(SAID, heard):.0%}", "|", heard)
else:
    print("Set MY_VOICE to a file you uploaded, then run this cell again.")

**Kahoot: which gives the most accurate transcript?** Four clips of the exact same sentence --
an ideal recording and three noisy versions. The facilitator plays each one and the room votes on
the ranking, best to worst, before the answer key cell below is run.


In [ ]:
NOISY_CLIPS = [
    ("1. ideal", "noisy_1_ideal.wav"),
    ("2. background hiss", "noisy_2_hiss.wav"),
    ("3. poor microphone", "noisy_3_poor_mic.wav"),
    ("4. a second voice underneath", "noisy_4_second_voice.wav"),
]

print("Vote now: which order, best to worst, before running the next cell?")
print()
for label, filename in NOISY_CLIPS:
    print(label)
    display(Audio(f"audio/{filename}"))


**Answer key.** Measured live, same as every other number in this notebook -- not assumed.

<details><summary>More detail: the numbers, by Whisper model size</summary>

The percentages depend strongly on the Whisper model size -- `MODEL` in section 1 (`base.en`,
`small.en` or `medium.en`), not a fixed property of the clips. All three rows below are CPU runs
(no GPU available when this was built):

| model | ideal | second voice | hiss | poor mic |
|---|---|---|---|---|
| `base.en` | 100% | 61% | 52% | 35% |
| `small.en` (this notebook's GPU default) | 100% | 87% | 83% | 22% |
| `medium.en` | 100% | 91% | 83% | 35% |

This order -- ideal, then the second voice underneath, then hiss, then the poor microphone --
held on the three sizes we tried. The exact percentages are only for this machine's run; run the
cell below to get yours.

</details>

In [ ]:
print("running Whisper on all four clips, about 20s total on CPU...")
for label, filename in NOISY_CLIPS:
    heard = hear(whisper.load_audio(f"audio/{filename}"))
    print(f"{label:32s} {word_accuracy(SAID, heard):.0%}  | {heard}")


### 2b. Say it yourself (live mic) *(optional, time permitting -- about 1 minute)*
**Predict:** if real background noise gets mixed into your recording, how much do you think
Whisper's accuracy will drop -- a little, or a lot?

In [ ]:
import live_mic

SAID_LIVE, clear_live_path, used_fallback = live_mic.say("block2_clear", model, seconds=5)
print("heard:", SAID_LIVE)

clear_live = whisper.load_audio(clear_live_path)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
t = np.arange(len(clear_live)) / 16000
ax1.plot(t, clear_live, linewidth=0.5)
ax1.set_ylabel("amplitude")
ax1.set_title("Your waveform")
with np.errstate(divide="ignore"):
    ax2.specgram(clear_live, Fs=16000, cmap="magma")
ax2.set_ylabel("pitch (Hz)")
ax2.set_xlabel("time (s)")
ax2.set_title("Your spectrogram")
plt.tight_layout()
plt.show()

**Try it:** say the same sentence again, this time with real background noise -- tap the
desk, rustle paper, or have music playing nearby.

In [ ]:
from audio_check import word_accuracy

NOISY_LIVE, noisy_live_path, _ = live_mic.say("block2_noisy", model, seconds=5)
print("heard:", NOISY_LIVE)
noisy_accuracy = word_accuracy(SAID_LIVE, NOISY_LIVE)
print(f"accuracy vs your own clear recording: {noisy_accuracy:.0%}")
if noisy_accuracy >= 1.0:
    print("Whisper coped with that noise -- try something louder (tap the desk harder, play music) to see it drop.")

**Why:** the exact mix of sounds you just made is what Whisper reads -- add anything else to
it (noise, a bad mic, an accent it hasn't heard much of) and accuracy OFTEN drops, same as cells
15-18 showed on the sample recording -- though a short, clear sentence can survive quite a lot of
noise and still come back 100%, which is its own useful data point. This is YOUR voice and YOUR
noise, not a canned example.

**Try it:** record one word on its own, then hear it played backwards.

In [ ]:
WORD_LIVE, word_live_path, _ = live_mic.say("block2_backward", model, seconds=3, minimum_words=1)
print("heard:", WORD_LIVE)

word_samples = whisper.load_audio(word_live_path)
print("forwards:")
display(Audio(word_samples, rate=16000))
print("backwards:")
display(Audio(word_samples[::-1].copy(), rate=16000))
backward_heard = model.transcribe(word_samples[::-1].copy(), language="en")["text"].strip()
print("Whisper on the reversed clip:", repr(backward_heard) if backward_heard else "(nothing)")

**Why:** order matters to Whisper (and to the RNN two sections below) -- reversed, your own
word stops being a word Whisper recognises, same as the forwards/backwards clip in section 1.

## 3. RNNs: reading words as a sequence (Agenda block 3, 15 min)
It reads one **word** at a time and carries a small "summary" (its memory) forward:
`new memory = mix of (this word) + (old memory)`, written as `h = tanh(W_x.x + W_h.h_prev + b)`

<details><summary>More detail</summary>

(Whisper looks at its spectrogram in time order too, though it's a different kind of
model (see section 1) -- the same "order matters" idea as the reversed word, just for
sound instead of words.)

</details>

In [ ]:
import numpy as np
from lstm_model import model_words

sentence = "Can you follow up with catering by Friday?"
words = model_words(sentence)
vocab = {w: i for i, w in enumerate(sorted(set(words)))}
numbers = [vocab[w] for w in words]

print("words  :", words)
print("numbers:", numbers, " (each word -> its own number, via a vocabulary)")

# Step a tiny RNN over these words, one at a time, to show "reads one word at a time" --
# toy random weights and embeddings, not the trained model (that comes in section 4).
rng = np.random.default_rng(0)
dim = 4
toy_embeddings = rng.normal(size=(len(vocab), dim))
W_x_toy, W_h_toy, b_toy = rng.normal(scale=0.5, size=(dim, dim)), rng.normal(scale=0.5, size=(dim, dim)), np.zeros(dim)

print()
print("Stepping the RNN one word at a time:")
memory = np.zeros(dim)
for word, number in zip(words, numbers):
    x = toy_embeddings[number]
    memory = np.tanh(W_x_toy @ x + W_h_toy @ memory + b_toy)
    print(f"  after {word!r:10s} memory = {np.round(memory, 2)}")


### 3b. Watch it read YOUR sentence (live mic)
**Predict:** will the memory update at every single word you say, or only some of them?

**Try it:** say any sentence, 5-15 words, at a normal pace.

In [ ]:
import toy_memory

print(f"Toy memory setting: keeps {toy_memory.KEEP_RATE:.0%} of the old memory at every word "
      f"(a stated number, not learned -- see notebook/toy_memory.py).")

SENTENCE_LIVE, sentence_live_path, _ = live_mic.say("block3_sentence", model, seconds=5)
print("heard:", SENTENCE_LIVE)
live_words = model_words(SENTENCE_LIVE)

h = np.zeros(8)
mem_changes = []
for w in live_words:
    h_new = toy_memory.step(live_mic.word_vector(w), h)
    mem_changes.append(float(np.abs(h_new - h).mean()))
    h = h_new

plt.figure(figsize=(8, 3))
plt.bar(range(len(live_words)), mem_changes)
plt.xticks(range(len(live_words)), live_words, rotation=45, ha="right")
plt.ylabel("change in memory (mean |h_t - h_t-1|)")
plt.title("How much the toy memory changes at each of YOUR words")
plt.tight_layout()
plt.show()
print(f"The memory changed at every one of your {len(live_words)} words -- no two words updated it by the same amount.")

**Why:** this plots the CHANGE at each word (`|h_t - h_t-1|`), not the memory's size -- the size
can look similar word to word even while the memory itself keeps moving in a new direction, which
is the actual claim ("reads one word at a time"). `toy_memory` (used again in blocks 4b and 5b
below) is a deliberately simple stand-in: `h = keep_rate * h_prev + (1 - keep_rate) * word_vector`,
with `live_mic.word_vector` standing in for a learned embedding -- nothing here has been trained.

### YOUR TURN (Part A): one step of an RNN (Agenda block 4, Hands-on A, 10 min)
At every step the RNN mixes **the new input** with **its old memory**:

`new memory = tanh( W_x × x  +  W_h × old memory  +  b )`

In numpy `×` is `@`, and tanh is `np.tanh`. Write that one line.

*`x` = this word, as numbers. `h_prev` = the memory so far. `W_x`, `W_h` = numbers the model*
*has learned (how much each one matters). `b` = a small nudge.*


In [ ]:
import numpy as np, hands_on

def rnn_step(x, h_prev, W_x, W_h, b):
    # TODO: return the new memory (the line above, in numpy)
    return None

hands_on.rnn_step = rnn_step
hands_on.MEMORY_ON = True
hands_on.part_a()

<details><summary>Stuck? (click to expand)</summary>

Verified by actually making each mistake and running it:

- **A 2x2 grid instead of a list of numbers** -- you used `*` somewhere `@` should be
  (`W_h * h_prev` multiplies element by element instead of mixing the memory together).
- **A number outside -1..1** (like `-3.15`) -- `tanh` is missing.
- **Close, but not quite** -- check you added `+ b` at the end.
- **`returned None`** -- the line still ends with `return None`; replace it with your formula.

</details>


Now **break the memory**. Two steps, both needed: delete `+ W_h @ h_prev` from your line above
(not just set a flag -- the term itself has to go), **and** set `hands_on.MEMORY_ON = False` in
that same cell. Run it again. What happens to the first word?

**Ignore the line printed above, "Now set MEMORY_ON = False and run again."** It's only half the
job -- it doesn't mention deleting the term, which is the step that actually matters. Do both
steps above, not just that one.

(If you'd rather see it without editing your own line first, the cell below does exactly this --
an "amnesia" version of `rnn_step`, wired in the same way.)


In [ ]:
# "Amnesia" version: the old-memory term is gone from the line itself, not just flagged off.
# Run this if you want to see the contrast before (or instead of) editing your own cell above.
def amnesia_rnn_step(x, h_prev, W_x, W_h, b):
    return np.tanh(W_x @ x + b)     # no + W_h @ h_prev at all

hands_on.rnn_step = amnesia_rnn_step
hands_on.MEMORY_ON = False
hands_on.part_a()


### 4b. Prove it with your own voice (live mic) *(bonus, if Part A above left time)*
**Predict:** with the memory term in, will "memory on" still tell two sentences apart by the
last word if they only differ in the first word? Will "memory off" (the amnesia version)?

**Try it:** read these two sentences aloud, one per cell below -- try to keep the words after the
first one the same, but it's fine if they drift (the code only ever compares your second
recording's first word against your first recording's own later words, so only the first word
differs either way):
- "Cats are sitting near the window this evening."
- "Dogs are sitting near the window this evening." 

In [ ]:
CATS_LIVE, _, _ = live_mic.say("block4_cats", model, seconds=5)
print("heard (cats):", CATS_LIVE)

In [ ]:
DOGS_LIVE, _, _ = live_mic.say("block4_dogs", model, seconds=5)
print("heard (dogs):", DOGS_LIVE)

cats_words = model_words(CATS_LIVE)
dogs_words_spoken = model_words(DOGS_LIVE)
dogs_words = toy_memory.first_word_pair(cats_words, dogs_words_spoken)
print(f"Comparing against: {' '.join(dogs_words)!r} -- your second recording's own first word "
      f"('{dogs_words[0]}') plus the first recording's own later words, so only the first word "
      f"differs even if Whisper heard the rest of the second recording differently.")

if dogs_words and cats_words and dogs_words[0] == cats_words[0]:
    print(f"Both recordings start with the same word ({cats_words[0]!r}) -- say a different "
          f"first word next time. Using a placeholder word for this run so the plot still works.")
    dogs_words = ["different"] + dogs_words[1:]

n_live = len(cats_words)
gaps_on = toy_memory.gap_curve(cats_words, dogs_words, live_mic.word_vector, keep_rate=toy_memory.KEEP_RATE)
gaps_off = toy_memory.gap_curve(cats_words, dogs_words, live_mic.word_vector, keep_rate=0.0)
pct_on = [g / gaps_on[0] * 100 for g in gaps_on]
pct_off = [g / gaps_off[0] * 100 for g in gaps_off]

plt.figure(figsize=(8, 3))
plt.plot(range(1, n_live + 1), pct_on, marker="o", label=f"memory ON (keeps {toy_memory.KEEP_RATE:.0%}/word)")
plt.plot(range(1, n_live + 1), pct_off, marker="o", label="memory OFF (amnesia)")
plt.xlabel("word")
plt.ylabel("% of word 1's difference remaining")
plt.title("Does it still remember your first word was different?")
plt.legend()
plt.tight_layout()
plt.show()
print(f"memory ON:  {pct_on[-1]:.1f}% left by word {n_live} -- keeps exactly {toy_memory.KEEP_RATE:.0%} every word, never zero")
print(f"memory OFF: {pct_off[-1]:.1f}% left by word {n_live} -- exactly 0% from word 2 on, every time")

**Why:** `toy_memory`'s keep rate is a stated number (0.8, not learned) -- so the ON curve's
decay is exact, not just usually true: it loses exactly 20% of what it's carrying every word,
forever, on any sentence. Memory OFF (keep rate 0) has no history term at all, so it is exactly
0% from word 2 on -- every time, not just on average. The gap you see (ON shrinking, OFF flat at
zero) is guaranteed by the formula itself, now true on sentences you actually said, not just the
one in the script.

### YOUR TURN: write your own first-word-swap pair
Write two sentences that differ in ONLY the first word (like the Cats/Dogs pair above). The
check below confirms it before using them.

In [ ]:
MY_SENTENCE_A = "Cats are sitting by the door tonight."   # YOUR TURN: write your own pair here
MY_SENTENCE_B = "Dogs are sitting by the door tonight."

my_words_a = model_words(MY_SENTENCE_A)
my_words_b = model_words(MY_SENTENCE_B)

if len(my_words_a) != len(my_words_b):
    print(f"check: not yet -- {len(my_words_a)} words vs {len(my_words_b)} words; match the word counts first.")
elif my_words_a and my_words_a[0] == my_words_b[0]:
    print("check: not yet -- the first word needs to actually be different.")
elif my_words_a[1:] != my_words_b[1:]:
    diff_at = next(i for i in range(1, len(my_words_a)) if my_words_a[i] != my_words_b[i])
    print(f"check: not yet -- word {diff_at + 1} also differs ({my_words_a[diff_at]!r} vs {my_words_b[diff_at]!r}); only the first word should.")
else:
    print("check: OK -- only the first word differs.")
    my_gaps = toy_memory.gap_curve(my_words_a, my_words_b, live_mic.word_vector)
    print(f"your pair: {my_gaps[-1] / my_gaps[0] * 100:.1f}% of the first word's effect left at the end")

<details><summary>Stuck? (click to expand)</summary>

- Different word counts -- add or remove a word from the shorter sentence until they match.
- A later word also differs -- copy sentence A, then change only its first word to get sentence B.
- This cell doesn't affect anything below it either way -- skip it and the rest of the notebook
  still runs.

</details>

### Why simple memory fades (Agenda block 5, 7 min)
Think of a game of broken telephone: pass a message through enough people and it's gone. An
RNN's memory fades the same way at every step. This is an **illustration** of that idea, not a
measured property of the trained model (which stacks 5 LSTM layers, not one plain RNN step).

(Part A's toy RNN above kept its own share each step, around 58% on its random weights -- the
dotted curve below is that exact number, measured in this cell, not typed in. The 0.49 curve is
what the untrained LSTM's own gates measured, before training, on this project's actual model.)

This is exactly why **long-range dependencies** matter: if an early line in a meeting sets up
something a much later line depends on, a memory that has already faded can't connect them. The
LSTM section below adds a dial for this -- training decides where to set it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Part A's toy RNN, replayed here with the correct line so this curve is always right,
# whether or not your own cell above is filled in yet -- same toy setup as Part A (seed 1).
toy_rng = np.random.default_rng(1)
toy_hidden = 8
toy_W_x, toy_W_h, toy_b = (toy_rng.normal(0, 0.5, (toy_hidden, 4)), toy_rng.normal(0, 0.25, (toy_hidden, toy_hidden)), np.zeros(toy_hidden))
toy_words_a = toy_rng.normal(size=(12, 4))
toy_words_b = toy_words_a.copy()
toy_words_b[0] = toy_rng.normal(size=4)
toy_h_a = toy_h_b = np.zeros(toy_hidden)
toy_gaps = []
for x_a, x_b in zip(toy_words_a, toy_words_b):
    toy_h_a = np.tanh(toy_W_x @ x_a + toy_W_h @ toy_h_a + toy_b)
    toy_h_b = np.tanh(toy_W_x @ x_b + toy_W_h @ toy_h_b + toy_b)
    toy_gaps.append(np.abs(toy_h_a - toy_h_b).sum())
toy_kept = np.mean([toy_gaps[k + 1] / toy_gaps[k] for k in range(len(toy_gaps) - 1) if toy_gaps[k] > 1e-12])

plt.figure(figsize=(7, 4))
n = np.arange(48)
for r, note in [(0.49, "untrained setting"), (0.95, "keeping 95%")]:
    retained = r ** n
    plt.plot(n, retained * 100, label=f"r = {r} ({note}): {retained[-1]:.1%} left at n=47")
plt.plot(n, (toy_kept ** n) * 100, linestyle=":", color="gray",
         label=f"Part A's toy RNN above (r = {toy_kept:.2f}, measured here): same fading")
plt.xlabel("steps (n)")
plt.ylabel("% of the original signal retained")
plt.title("Memory retention: r^n (an illustration, not measured)", fontsize=11)
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.ylim(0, 100)
plt.show()

print(f"0.49^47 = {0.49**47:.4%}   0.95^47 = {0.95**47:.1%}   toy RNN: r={toy_kept:.3f}, {toy_kept**47:.4%} left at n=47")


### 5b. Does a SHORT swap survive further than a LONG one? (live mic)
**Predict:** which will still show a bigger trace of the first word by the end -- a 4-word pair,
or a 15+-word pair?

**Try it:** read these two lines aloud, one per cell below, exactly as written:
- "Red cars drive fast." / "Blue cars drive fast." (4 words, first word swapped)

**Try it** *(bonus, if there's time)*: read these two longer lines aloud, one per cell below:
- "Monday mornings always feel quieter than every other weekday afternoon session we have had
  this term." / the same sentence starting with "Tuesday" instead (16 words, first word
  swapped)

In [ ]:
SHORT_A, _, _ = live_mic.say("block5_short_a", model, seconds=4)
print("heard (short a):", SHORT_A)

In [ ]:
SHORT_B, _, _ = live_mic.say("block5_short_b", model, seconds=4)
print("heard (short b):", SHORT_B)

In [ ]:
LONG_A, _, _ = live_mic.say("block5_long_a", model, seconds=8)
print("heard (long a):", LONG_A)

In [ ]:
LONG_B, _, _ = live_mic.say("block5_long_b", model, seconds=8)
print("heard (long b):", LONG_B)

short_words_a = model_words(SHORT_A)
short_words_b = toy_memory.first_word_pair(short_words_a, model_words(SHORT_B))
long_words_a = model_words(LONG_A)
long_words_b = toy_memory.first_word_pair(long_words_a, model_words(LONG_B))

if short_words_a and short_words_b and short_words_a[0] == short_words_b[0]:
    print(f"Short pair: both start with {short_words_a[0]!r} -- say a different first word next "
          f"time. Using a placeholder word for this run so the chart still works.")
    short_words_b = ["different"] + short_words_b[1:]
if long_words_a and long_words_b and long_words_a[0] == long_words_b[0]:
    print(f"Long pair: both start with {long_words_a[0]!r} -- say a different first word next "
          f"time. Using a placeholder word for this run so the chart still works.")
    long_words_b = ["different"] + long_words_b[1:]

short_gaps = toy_memory.gap_curve(short_words_a, short_words_b, live_mic.word_vector)
long_gaps = toy_memory.gap_curve(long_words_a, long_words_b, live_mic.word_vector)
short_pct = short_gaps[-1] / short_gaps[0] * 100
long_pct = long_gaps[-1] / long_gaps[0] * 100
short_below10 = toy_memory.words_below(short_gaps, 10)
long_below10 = toy_memory.words_below(long_gaps, 10)

print(f"Toy memory setting: keeps {toy_memory.KEEP_RATE:.0%} of the old memory at every word "
      f"(a stated number, not learned).")

plt.figure(figsize=(5, 3.5))
bars = plt.bar([f"short pair{chr(10)}({len(short_words_a)} words)", f"long pair{chr(10)}({len(long_words_a)} words)"],
               [short_pct, long_pct], color=["green", "red"])
plt.ylabel("% of the first word's difference left at the end")
plt.title("How much of your first word survives?")
plt.ylim(0, 100)
for bar, v in zip(bars, [short_pct, long_pct]):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 2, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()

short_note = f", falls below 10% after word {short_below10}" if short_below10 else " -- never falls below 10%"
long_note = f", falls below 10% after word {long_below10}" if long_below10 else " -- never falls below 10%"
print(f"short pair ({len(short_words_a)} words): {short_pct:.1f}% of the first word's effect left{short_note}")
print(f"long pair  ({len(long_words_a)} words): {long_pct:.2f}% of the first word's effect left{long_note}")

**Why:** `toy_memory`'s keep rate (0.8, stated above) means the first word's effect decays as
exactly `0.8 ** (word - 1)` -- a 4-word pair is guaranteed to keep about half, a 16-word pair is
guaranteed to be down near 3%, on any sentence, not just this one (checked over 500 random
first-word swaps: zero variance -- see `notebook/toy_memory.py`'s self-test). Fewer steps really
does leave more of the first word's effect; this is not the trained model, which stacks 5 LSTM
layers instead of one plain step (see the caveat on the illustration graph above).

### 5c. The headline: does a SHORT exchange remember better than a LONG conversation? (live mic)
**Predict:** if the very first thing said in a conversation matters later, will a short exchange
still "remember" it? Will a long, sustained one?

**Try it:** say a SHORT exchange, 2-3 turns, ~2s pause between turns, where the FIRST turn holds
one key fact (for example "The meeting is on Friday at three."):

In [ ]:
from transcriber import segments_to_transcript

SHORT_CONV, short_conv_path, _ = live_mic.say("headline_short_conv", model, seconds=15, minimum_words=6)
print("heard (flat):", SHORT_CONV)

short_result = model.transcribe(short_conv_path, word_timestamps=True, language="en")
short_words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                  for seg in short_result["segments"] for w in seg.get("words", [])]
short_turn_text = segments_to_transcript(short_words_ts, pause_gap=1.2)
short_turn_lines = [l for l in short_turn_text.strip().split(chr(10)) if l]
short_conv_turns = [(f"Turn{i + 1}", line.split(": ", 1)[1]) for i, line in enumerate(short_turn_lines)]
print("turns heard:", short_conv_turns)

**Try it:** now a LONG, sustained conversation, 8+ turns (about 40-60s) -- same idea, the FIRST
turn holds one key fact, then keep talking about other things:

In [ ]:
from lstm_model import tokenize_transcript

LONG_CONV, long_conv_path, _ = live_mic.say("headline_long_conv", model, seconds=50, minimum_words=20)
print("heard (flat):", LONG_CONV)

long_result = model.transcribe(long_conv_path, word_timestamps=True, language="en")
long_words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                 for seg in long_result["segments"] for w in seg.get("words", [])]
long_turn_text = segments_to_transcript(long_words_ts, pause_gap=1.2)
long_turn_lines = [l for l in long_turn_text.strip().split(chr(10)) if l]
long_conv_turns = [(f"Turn{i + 1}", line.split(": ", 1)[1]) for i, line in enumerate(long_turn_lines)]
print("turns heard:", long_conv_turns)

short_tokens, _ = tokenize_transcript(short_conv_turns)
long_tokens, _ = tokenize_transcript(long_conv_turns)
ALT_FIRST_LINE = model_words("The meeting is on Monday at nine.")

short_tokens_b = toy_memory.first_line_pair(short_tokens, ALT_FIRST_LINE)
long_tokens_b = toy_memory.first_line_pair(long_tokens, ALT_FIRST_LINE)

short_gaps = toy_memory.gap_curve(short_tokens, short_tokens_b, live_mic.word_vector)
long_gaps = toy_memory.gap_curve(long_tokens, long_tokens_b, live_mic.word_vector)
# Normalize from the END of line 1 (not word 1) -- a line-level swap can share
# its first word or two with the alternate line ("The meeting is on Friday..."
# vs "...on Monday..."), so the gap can be 0 right at word 1 even though the
# lines differ; the gap once the whole line has been heard is the real "100%".
short_ref = short_gaps[toy_memory.line_boundary_index(short_tokens)]
long_ref = long_gaps[toy_memory.line_boundary_index(long_tokens)]
short_pct_conv = short_gaps[-1] / short_ref * 100
long_pct_conv = long_gaps[-1] / long_ref * 100

print(f"Toy memory (keep {toy_memory.KEEP_RATE:.0%}/word, same stated setting as blocks 3-5 above).")
plt.figure(figsize=(5, 3.5))
bars = plt.bar(["short exchange", "long conversation"], [short_pct_conv, long_pct_conv], color=["green", "red"])
plt.ylabel("% of line 1's fact still in memory at the end")
plt.title("Does the toy RNN remember line 1 by the end?")
plt.ylim(0, 100)
for bar, v in zip(bars, [short_pct_conv, long_pct_conv]):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 2, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()
print(f"short exchange ({len(short_tokens)} tokens): {short_pct_conv:.1f}% of line 1's fact left")
print(f"long conversation ({len(long_tokens)} tokens): {long_pct_conv:.2f}% of line 1's fact left")

**Why:** `toy_memory`'s keep rate (0.8, same stated setting as above) means a short exchange
keeps a clear share of line 1's fact, but a long conversation runs it down to almost nothing --
exactly the "fewer steps leaves more" story from 5b, just on a whole conversation instead of one
sentence. Block 8 revisits this same recording through the toy LSTM's dial; block 9 checks
whether it's still true for the TRAINED model.

## 4. LSTMs: an eraser you can set (Agenda block 6, 20 min)
An RNN's memory fades a little at every step, at one fixed rate. An LSTM adds a **dial**: its
memory is a **notebook** (the cell state) with three switches at every step -- the **eraser**
(forget gate, `f`), the **pencil** (input gate, `i`), and the **voice** (output gate, `o`).
`new notebook = (what you keep) x (old notebook) + (what you add) x (new note)`, written as
`c = f*c_prev + i*g`

<details><summary>More detail: the hourglass shape, and the end-of-turn token</summary>

Training sets the dial; Hands-on B below lets you turn it yourself. Our model stacks 5 of these
layers, like an hourglass: the middle layer looks at four words at a time (folded together) -- an
architecture idea, not something this trained classifier measures or proves on its own. A
`<eot>` token between turns is the "speaker changed" bookmark -- the model's only signal that
someone new is talking.

</details>

In [ ]:
from lstm_model import run_pipeline, load_trained_model, tokenize_transcript, pad_to_multiple_of_4

turns, output, turn_ends = run_pipeline(transcript=transcript)
trained = load_trained_model()
vocab = {w: i for i, w in enumerate(trained["vocab"])}
tokens, _ends = tokenize_transcript(turns)
x = trained["embeddings"][[vocab.get(w, 0) for w in tokens]]
_hourglass_out, hourglass_layers = trained["model"].forward(pad_to_multiple_of_4(x))

print("The hourglass: 5 layers, each with its own units and time steps (this notebook's own meeting):")
print(f"  {'layer':<9}{'units':>7}{'time steps':>13}")
for name, out in hourglass_layers.items():
    steps, units = out.shape
    print(f"  {name:<9}{units:>7}{steps:>13}")

layer_names = list(hourglass_layers.keys())
layer_units = [out.shape[1] for out in hourglass_layers.values()]
plt.figure(figsize=(6, 2.5))
plt.bar(layer_names, layer_units)
plt.ylabel("units")
plt.title("The hourglass shape: units per layer")
plt.show()


**Watch the gates.** The eraser, pencil and voice are numbers the model actually computes --
not just names. Layer 1 of the trained model (one ensemble member), replayed by hand from its
shipped weights over the hook's own exchange, "Can you book the room? / Sure.":

*(Caveat: layer 1 only, one of five ensemble members, the mean of its 8 units -- a sketch of
what the gates do, not the whole model.)*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lstm_model import TURN_BOUNDARY_TOKEN, tokenize_transcript

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

gate_example = [("Mei", "Can you book the room?"), ("Ravi", "Sure.")]
gate_tokens, _gate_ends = tokenize_transcript(gate_example)
gate_ids = [vocab.get(w, 0) for w in gate_tokens]
gate_x_seq = trained["embeddings"][gate_ids]
layer1 = trained["model"].layer1

h, c = np.zeros(layer1.hidden_size), np.zeros(layer1.hidden_size)
erasers, pencils, voices = [], [], []
for x in gate_x_seq:
    combined = np.concatenate([h, x])
    f = sigmoid(layer1.W_f @ combined + layer1.b_f)
    i_gate = sigmoid(layer1.W_i @ combined + layer1.b_i)
    c_tilde = np.tanh(layer1.W_c @ combined + layer1.b_c)
    c = f * c + i_gate * c_tilde
    o = sigmoid(layer1.W_o @ combined + layer1.b_o)
    h = o * np.tanh(c)
    erasers.append(f.mean())
    pencils.append(i_gate.mean())
    voices.append(o.mean())

x_pos = np.arange(len(gate_tokens))
width = 0.25
plt.figure(figsize=(9, 3))
plt.bar(x_pos - width, erasers, width, label="eraser (forget)")
plt.bar(x_pos, pencils, width, label="pencil (input)")
plt.bar(x_pos + width, voices, width, label="voice (output)")
plt.xticks(x_pos, gate_tokens, rotation=45, ha="right")
plt.ylabel("gate value (mean of 8 units)")
plt.title("Layer 1's gates, one word at a time (member 0)")
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.tight_layout()
plt.show()

for tok, f_m, i_m, o_m in zip(gate_tokens, erasers, pencils, voices):
    marker = "  <- speaker changed" if tok == TURN_BOUNDARY_TOKEN else ""
    print(f"{tok:10s} eraser={f_m:.2f}  pencil={i_m:.2f}  voice={o_m:.2f}{marker}")


`<eot>` is the only signal the model gets that a new speaker is talking -- there is no speaker
name in the input (`tokenize_transcript` drops it; `prototype/demo.py` only puts the real names
back into the minutes afterwards). Here it is in a 2-turn exchange:


In [ ]:
from lstm_model import TURN_BOUNDARY_TOKEN, tokenize_transcript

example = [("Alex", "Can you follow up with catering by Friday?"), ("Priya", "Yes, I'll email them today.")]
example_tokens, example_ends = tokenize_transcript(example)
print(example_tokens)
print(f"{TURN_BOUNDARY_TOKEN!r} marks \"speaker changed\" between the two turns "
      f"(turn_end_indices = {example_ends}).")


In [ ]:
import numpy as np
from lstm_model import fold_time, unfold_time
x = np.arange(16.0).reshape(8, 2)
print("8 steps of 2 numbers, folded:", fold_time(x).shape, "(4 steps of 4); unfolded back is identical:", np.array_equal(unfold_time(fold_time(x)), x))

### 6b. Watch the gates on YOUR turns (live mic)
**Predict:** where will the eraser dip the most -- in the middle of a sentence, or right where
the speaker changes?

**Try it:** say two short turns, with a clear ~2s pause between them, for example a question and
a one-word answer (you can play both people).

In [ ]:
from transcriber import segments_to_transcript
from lstm_model import tokenize_transcript, TURN_BOUNDARY_TOKEN

GATE_LIVE, gate_path, _ = live_mic.say("block6_gates", model, seconds=8)
print("heard (flat):", GATE_LIVE)

gate_result = model.transcribe(gate_path, word_timestamps=True, language="en")
gate_words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                  for seg in gate_result["segments"] for w in seg.get("words", [])]
gate_turn_text = segments_to_transcript(gate_words_ts, pause_gap=1.2)
gate_turn_lines = [l for l in gate_turn_text.strip().split(chr(10)) if l]
gate_live_turns = [(f"Turn{i + 1}", line.split(": ", 1)[1]) for i, line in enumerate(gate_turn_lines)]
print("turns heard:", gate_live_turns)

if len(gate_live_turns) < 2:
    print("I heard it as one turn -- leave a 2s pause before the other person speaks, then run this cell again.")

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

gate_live_tokens, _gate_live_ends = tokenize_transcript(gate_live_turns)
gate_live_ids = [vocab.get(w, 0) for w in gate_live_tokens]
gate_live_x_seq = trained["embeddings"][gate_live_ids]
live_layer1 = trained["model"].layer1   # the TRAINED model's own layer 1 (member 0), not a toy

h, c = np.zeros(live_layer1.hidden_size), np.zeros(live_layer1.hidden_size)
live_erasers, live_pencils = [], []
for x in gate_live_x_seq:
    combined = np.concatenate([h, x])
    f = sigmoid(live_layer1.W_f @ combined + live_layer1.b_f)
    i_gate = sigmoid(live_layer1.W_i @ combined + live_layer1.b_i)
    c_tilde = np.tanh(live_layer1.W_c @ combined + live_layer1.b_c)
    c = f * c + i_gate * c_tilde
    h = sigmoid(live_layer1.W_o @ combined + live_layer1.b_o) * np.tanh(c)
    live_erasers.append(f.mean())
    live_pencils.append(i_gate.mean())

x_pos = np.arange(len(gate_live_tokens))
width = 0.35
plt.figure(figsize=(9, 3))
plt.bar(x_pos - width / 2, live_erasers, width, label="eraser (forget)")
plt.bar(x_pos + width / 2, live_pencils, width, label="pencil (input)")
plt.xticks(x_pos, gate_live_tokens, rotation=45, ha="right")
plt.ylabel("gate value (mean of 8 units)")
plt.title("Layer 1's gates over YOUR two turns (trained model, member 0)")
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8, borderaxespad=0)
plt.tight_layout()
plt.show()

eot_positions = [i for i, t in enumerate(gate_live_tokens) if t == TURN_BOUNDARY_TOKEN]
if eot_positions:
    eot_eraser = live_erasers[eot_positions[0]]
    other = [e for i, e in enumerate(live_erasers) if i not in eot_positions]
    print(f"eraser at <eot> = {eot_eraser:.2f}; everywhere else: {min(other):.2f}-{max(other):.2f}")
else:
    print("(no <eot> heard -- say two clearly separate turns with a ~2s pause between them)")

**Why:** this is the TRAINED model's own layer 1 (member 0), the same technique the static
"watch the gates" cell above uses -- not a toy, but still only one layer of one ensemble member,
so it's a sketch, not the whole model's decision. *(Caveat, checked on 1000 random two-turn
exchanges: the eraser is lowest exactly at `<eot>` in about 82% of them, and below the median in
all of them -- say "usually dips at the speaker change", not "always".)*

**Try it:** now put your Cats/Dogs-style pair from 4b (or the shipped fallback pair, if you
skipped 4b) through the toy LSTM below, at three different knob settings -- this part IS still an
untrained toy, on purpose, to isolate what the knob alone does.

In [ ]:
from lstm_model import SimpleLSTM

# Reuses cats_words/dogs_words from 4b if it ran (bonus) -- already built so only
# the first word differs. If 4b was skipped, get the pair now instead, so this
# cell never depends on a skipped bonus cell.
if "cats_words" not in globals() or "dogs_words" not in globals():
    print("4b wasn't run -- getting the Cats/Dogs pair now instead.")
    _cats_live, _, _ = live_mic.say("block4_cats", model, seconds=5)
    _dogs_live, _, _ = live_mic.say("block4_dogs", model, seconds=5)
    cats_words = model_words(_cats_live)
    dogs_words = toy_memory.first_word_pair(cats_words, model_words(_dogs_live))
    if cats_words and dogs_words and cats_words[0] == dogs_words[0]:
        print(f"Both start with the same word ({cats_words[0]!r}) -- using a placeholder so this still works.")
        dogs_words = ["different"] + dogs_words[1:]

n_l = len(cats_words)

def lstm_notebook_gap(words_a, words_b, knob):
    layer = SimpleLSTM(8, 8, seed=5)
    layer.b_f = layer.b_f + knob
    ha = hb = ca = cb = np.zeros(8)
    gaps = []
    for wa, wb in zip(words_a, words_b):
        xa, xb = live_mic.word_vector(wa), live_mic.word_vector(wb)
        ha, ca = layer.step(xa, ha, ca)
        hb, cb = layer.step(xb, hb, cb)
        gaps.append(np.abs(ca - cb).sum())
    return gaps

knob_gaps = {knob: lstm_notebook_gap(cats_words[:n_l], dogs_words[:n_l], knob) for knob in (-6, 0, 6)}

plt.figure(figsize=(8, 3))
for knob, gaps in knob_gaps.items():
    plt.plot(range(1, n_l + 1), gaps, marker="o", label=f"knob {knob:+d}")
plt.xlabel("word")
plt.ylabel("|notebook(cats) - notebook(dogs)|")
plt.title("Your first-word swap through the toy LSTM, three knob settings")
plt.legend()
plt.tight_layout()
plt.show()

for knob, gaps in knob_gaps.items():
    note = "erases fastest" if knob == -6 else ("keeps longest" if knob == 6 else "default")
    print(f"knob {knob:+d} ({note}): gap at word 2 = {gaps[1]:.2f}, by word {n_l} = {gaps[-1]:.4f}")

**Why:** the eraser knob decides how long your own first word survives in the notebook --
more negative erases it faster, more positive keeps it longer, on the exact words you spoke, not
just the scripted Cats/Dogs pair from section 4.

### Teaching it, and catching it out (Agenda block 7, 10 min)
`label -> prediction -> loss -> update`, repeated over many lines. How wrong is measured as
`loss = -log(p(the model gave the correct label))`: confident and right is a tiny loss, confident
and wrong is a large one.

<details><summary>More detail: the analogy</summary>

Training works like marking practice papers: the model sees a meeting line with the right
label, guesses, is told how wrong it was, and adjusts a little. Worked example, on this
notebook's own meeting:

</details>

In [ ]:
from lstm_model import label_probabilities
from meeting_minutes import label_turns_stub

gold = label_turns_stub(turns)
probs = label_probabilities(turns, trained)

for (speaker, text), g, p in zip(turns, gold, probs):
    if "email them today" in text or "Venue is confirmed" in text:
        correct = g["category"]
        p_correct = p[trained["labels"].index(correct)]
        print(text)
        print(f"  gold = {correct}, p(correct) = {p_correct:.4f}, loss = {-np.log(p_correct):.2f}")
        print()


The model is confident AND right on "Yes, I'll email them today." (tiny loss), and confident
AND **wrong** on "Venue is confirmed for the 14th..." (a decision stated as a plain fact) -- a
large loss, exactly the kind of line that pulls training the most.

<details><summary>More detail: keyword rules vs. the trained model, across all held-out meetings</summary>

**Testing it on lines it has never seen.** A simple keyword checklist (does the line contain
"confirmed", "agreed", ...?) versus the trained model, on a held-out meeting -- a student
council, never used in training. This is the team's **best-case** held-out meeting for the
trained model, not a typical one: across all 17 held-out meetings (`holdout_data.py`) the
trained LSTM scores 131/149 (0.879) and 104/122 (0.852) (`python prototype/run_tests.py`
check k) against the keyword checklist's 111/149 (0.745) and 93/122 (0.762) (a per-meeting
count over `holdout_data.py`, not printed by any existing script) -- a real, solid improvement,
but nowhere near 17/18. On 3 of those 17 meetings the keyword checklist actually scores
*better* than the trained model.

</details>

In [ ]:
from pipeline import load_text
from meeting_minutes import label_turns_keywords, label_turns_model

held_out_text, held_out_gold = load_text(f"{PROTO}/demo_student_council.txt")
held_out_turns, held_out_output, held_out_ends = run_pipeline(transcript=held_out_text)
kw_labels = label_turns_keywords(held_out_turns)
lstm_labels = label_turns_model(held_out_turns, held_out_output, held_out_ends)

kw_right = sum(k["category"] == g for k, g in zip(kw_labels, held_out_gold))
lstm_right = sum(l["category"] == g for l, g in zip(lstm_labels, held_out_gold))
print(f"keyword rules: {kw_right}/{len(held_out_turns)} turns right")
print(f"trained LSTM:  {lstm_right}/{len(held_out_turns)} turns right")
print()

for (speaker, text), g, l in zip(held_out_turns, held_out_gold, lstm_labels):
    if l["category"] != g:
        print(f"LSTM miss: {text!r}  (gold {g}, got {l['category']})")


The labelling task mostly needs about one previous turn (a bare "Sure." after a request) --
not a long memory. **We don't claim the trained LSTM remembers further back than a plain RNN**
on this task; the long-range-dependency idea from the memory-fade graph above is illustrated by
the toy experiments (Hands-on A and B), not measured by this classifier.

And as seen above: a decision reported as a plain fact ("Venue is confirmed...", "No refreshments
this time...") is where the trained model is weakest -- interestingly, the simple keyword
checklist actually catches "Venue is confirmed" (it contains the word "confirmed"), while the
trained model, which should in principle do better, misses it. Neither approach is perfect; that
is the honest result, not a reason to trust one blindly over the other.


### 7b. Catch it out yourself (live mic)
**Predict:** will the model do better on a request, a proposal, or a plain fact stated as fact (no
"let's" or "agree" to catch it on)?

**Try it:** say one of each, one per cell below.

In [ ]:
REQUEST_LIVE, _, _ = live_mic.say("block7_request", model, seconds=5)
print("heard:", REQUEST_LIVE)

In [ ]:
PROPOSAL_LIVE, _, _ = live_mic.say("block7_proposal", model, seconds=5)
print("heard:", PROPOSAL_LIVE)

In [ ]:
FACT_LIVE, _, _ = live_mic.say("block7_fact", model, seconds=5)
print("heard:", FACT_LIVE)

from meeting_minutes import label_turns_keywords

INTENDED = {"request": "action_item", "proposal": "decision", "plain fact": "decision"}

for kind, text in [("request", REQUEST_LIVE), ("proposal", PROPOSAL_LIVE), ("plain fact", FACT_LIVE)]:
    turns_live7 = [("You", text)]
    probs_live7 = label_probabilities(turns_live7, trained)[0]
    top_cat, top_p = max(zip(trained["labels"], probs_live7), key=lambda lp: lp[1])
    intended_cat = INTENDED[kind]
    intended_p = probs_live7[trained["labels"].index(intended_cat)]
    loss7 = -np.log(intended_p)   # the loss IF this is the right reading -- catching it out means this is large
    kw_cat = label_turns_keywords(turns_live7)[0]["category"]
    print(f"{kind:10s}: model -> {top_cat} ({top_p:.0%})   "
          f"loss if it should be {intended_cat!r} = {loss7:.2f}   keyword rule -> {kw_cat}")

**Why:** a request or a proposal usually carries a word the keyword rules (and the model)
can key on ("can you", "let's"). A plain fact stated as fact has neither -- that's the model's
documented weak spot (see `labelling_guide.md`), and it's visible right here on something YOU
said, not just the scripted "Venue is confirmed" example above.

**The "Sure." test -- back to the hook.** The opening exchanges were "Can you book the room?"
"Sure." and "Shall we move it to Friday?" "Sure." -- the room gets both instantly because they
remember the line before. Same two words, run through the model three ways:


In [ ]:
sure_cases = [
    [("Ravi", "Sure.")],
    [("Mei", "Can you book the room?"), ("Ravi", "Sure.")],
    [("Mei", "Shall we move it to Friday?"), ("Ravi", "Sure.")],
]
for case in sure_cases:
    case_text = chr(10).join(f"{s}: {t}" for s, t in case)
    case_turns, case_output, case_ends = run_pipeline(transcript=case_text)
    case_probs = label_probabilities(case_turns, trained)[-1]
    lead_in = case[0][1] if len(case) > 1 else "(no lead-in)"
    probs_str = ", ".join(f"{l} {p:.2f}" for l, p in sorted(zip(trained["labels"], case_probs), key=lambda x: -x[1]))
    print(f"{lead_in:28s} -> {probs_str}")


"Sure." alone could be anything (the three scores are close). After the request it's
confidently an action item -- right. After the proposal it's confidently... **discussion** --
**wrong**: a bare agreement to a proposal is a decision (`labelling_guide.md`). The model reads
context, same as the room does, but it doesn't always land on the right answer from it. That is
the honest version of the hook's closing line, and it's the lesson for block 9: context changes
the model's answer, but doesn't guarantee it's correct.


### The same word, two different answers (live mic) -- back to the hook
**Predict:** "Sure." on its own could mean anything. Does the model read it differently depending
on what came right before it, the way the room did at the hook?

**Try it:** say these two short exchanges, each with a ~2s pause, one per cell below:
- "Can you book the room?" / (pause) / "Sure."
- "Shall we move it to Friday?" / (pause) / "Sure."

In [ ]:
SURE_REQ_LIVE, sure_req_path, _ = live_mic.say("sure_request", model, seconds=8)
print("heard:", SURE_REQ_LIVE)

In [ ]:
SURE_PROP_LIVE, sure_prop_path, _ = live_mic.say("sure_proposal", model, seconds=8)
print("heard:", SURE_PROP_LIVE)

def label_sure_exchange(path, pause_gap=1.2):
    result = model.transcribe(path, word_timestamps=True, language="en")
    words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                for seg in result["segments"] for w in seg.get("words", [])]
    turn_text = segments_to_transcript(words_ts, pause_gap=pause_gap)
    lines = [l for l in turn_text.strip().split(chr(10)) if l]
    turns_sure = [(f"Speaker{i % 2 + 1}", l.split(": ", 1)[1]) for i, l in enumerate(lines)]
    probs_sure = label_probabilities(turns_sure, trained) if len(turns_sure) > 1 else None
    return turns_sure, probs_sure

for lead_in, path in [("after the request", sure_req_path), ("after the proposal", sure_prop_path)]:
    turns_sure, probs_sure = label_sure_exchange(path)
    if len(turns_sure) < 2:
        print(f"{lead_in:20s}: I heard it as one turn -- leave a 2s pause before the other "
              f"person speaks, then try this prompt again.")
        continue
    found_sure = False
    for (speaker, text), p in zip(turns_sure, probs_sure):
        if text.strip().lower().rstrip(".") == "sure":
            top_cat, top_p = max(zip(trained["labels"], p), key=lambda lp: lp[1])
            print(f"{lead_in:20s}: 'Sure.' -> {top_cat} ({top_p:.0%})")
            found_sure = True
    if not found_sure:
        print(f"{lead_in:20s}: didn't hear a separate 'Sure.' turn -- leave a 2s pause before "
              f"it, then try again.")

**Why:** same spoken word, read two different ways depending on what you said right before
it -- same as the hook. Honest: on your own voice this may land differently than the scripted
version above (the model is small, and this is a genuine knife-edge case, not a sure thing); the
"Sure." test cell further up already showed the proposal-lead-in case is the one the model tends
to get wrong.

### YOUR TURN (Part B): the LSTM notebook update (Agenda block 8, Hands-on B, 20 min)
`new notebook = (how much to keep) × (old notebook) + (how much to add) × (the new note)`

`f` is the eraser, `i` is the pencil, `g` is the new note, `c_prev` is the old notebook. Write that one line.

In [ ]:
def lstm_cell_update(f, i, g, c_prev):
    # TODO: return the new notebook (the line above, in numpy)
    return None

hands_on.lstm_cell_update = lstm_cell_update
hands_on.FORGET_KNOB = 0          # try -6, -3, 0, 3, 6: negative erases more, positive keeps more
hands_on.part_b()

<details><summary>Stuck? (click to expand)</summary>

Verified by actually making each mistake and running it:

- **Bigger numbers than expected** -- you used `+` somewhere `*` should be
  (`f + c_prev` instead of `f * c_prev` keeps everything instead of erasing a share of it).
- **Close, but not quite** -- check the new note `g` is actually multiplied in
  (`i * g`, not just `i`).
- **`returned None`** -- the line still ends with `return None`; replace it with your formula.

</details>


Turn the knob to **-6** and **+6** and run the cell again. What happens to the first word, and to the trained model's score? Why do both extremes hurt it?

**All three settings, compared.** Run the knob at -6, 0 and +6 in one go: "keep a lot" (+6) is
**not** "better memory" -- the trained model learned its own eraser setting at 0, so pushing the
knob either way makes its labels worse.

<details><summary>More detail: why each extreme hurts, and the link to the retention graph</summary>

Look at both numbers each time: the toy gap at word 12, and the trained model's score on the
test meetings (-6, -3, 0, +3, +6 give 10/45, 16/45, 35/45, 23/45 and 22/45). *(This sweep runs
one of the five models in the ensemble -- the same one `hands_on.py` uses -- while the "Sure."
test above averages all five; the two cells aren't measuring quite the same thing.)*

- **-6 hurts** because the model erases everything, including the earlier line a reply like
  "Sure." needs (the hook's whole point) -- the toy gap dies at word 2.
- **+6 hurts** because the notebook barely erases anything: old content keeps piling up and
  swamps the new word each step, so the toy gap doesn't fade -- it actually **grows** from
  0.17 to 0.37, the opposite of fading, because every new note adds on top of notes that were
  never cleared.

The dial from above is this same forget gate: a knob of 0 sits at `sigmoid(0) = 0.5`, keeping
about half each step -- the 0.49 curve in the retention graph. A knob of +3 sits at
`sigmoid(3) = 0.953`, keeping about 95% -- the 0.95 curve. That's the link between the two
graphs: the retention illustration above is what this exact knob is tuning.

</details>

In [ ]:
for knob in (-6, 0, 6):
    hands_on.FORGET_KNOB = knob
    hands_on.part_b()
    print()
hands_on.FORGET_KNOB = 0   # leave the knob back at its trained setting


### 8b. Turn the knob on YOUR exchange (live mic)
**Predict:** will turning the knob change the trained model's label on an exchange you made up,
the same way it changed the test-meeting score above?

**Try it:** say a short two-turn exchange, a question and a short answer, ~2s pause between
turns -- you can reuse "Can you confirm the venue?" / "Sure." or make up your own.

In [ ]:
EXCHANGE_LIVE, exchange_path, _ = live_mic.say("block8_exchange", model, seconds=8)
print("heard (flat):", EXCHANGE_LIVE)

turns_live8, _ = label_sure_exchange(exchange_path)
print("turns:", turns_live8)

if len(turns_live8) < 2:
    print("I heard it as one turn -- leave a 2s pause before the other person speaks, then try this prompt again.")
else:
    def predict_with_knob(turns, knob):
        """Knob applied to ALL 5 ensemble members, then averaged -- same ensembling
        `label_probabilities` uses everywhere else, so this cell agrees with the
        Sure. test and every other trained-model cell instead of contradicting them."""
        tokens, ends = tokenize_transcript(turns)
        all_probs = []
        for member in trained["members"]:
            original_b_f = {n: getattr(member["model"], f"layer{n}").b_f.copy() for n in range(1, 6)}
            for n in range(1, 6):
                layer_n = getattr(member["model"], f"layer{n}")
                layer_n.b_f = original_b_f[n] + knob
            ids = [vocab.get(w, 0) for w in tokens]
            x = member["embeddings"][ids]
            out, _ = member["model"].forward(pad_to_multiple_of_4(x))
            scores = out[:len(x)][ends] @ member["head_W"].T + member["head_b"]
            e = np.exp(scores - scores.max(axis=1, keepdims=True))
            all_probs.append(e / e.sum(axis=1, keepdims=True))
            for n in range(1, 6):
                getattr(member["model"], f"layer{n}").b_f = original_b_f[n]   # restore right away
        return np.mean(all_probs, axis=0)

    for knob in (-6, 0, 6):
        probs8 = predict_with_knob(turns_live8, knob)
        labels8 = [max(zip(trained["labels"], p), key=lambda lp: lp[1]) for p in probs8]
        print(f"knob {knob:+d}:", ", ".join(f"{s}: {lbl} ({p:.0%})" for (s, t), (lbl, p) in zip(turns_live8, labels8)))

**Why:** unlike the static sweep above (one ensemble member, same as `hands_on.py`'s
convention), this cell applies the knob to **all five** members and averages -- the same
ensembling `label_probabilities` uses everywhere else, so it agrees with the Sure. test instead
of contradicting it. The model learned its own eraser setting at 0, so the knob moving either way
can flip a label that was right at 0 -- on YOUR words, not just the test set's.

### 8c. Same test, through the toy LSTM's keep dial (live mic recordings from 5c)
**See it (contrast):** the same short/long conversations through the toy LSTM at a "keep" dial of
+3 -- still an untrained, stated setting, not something our trained model learned.

In [ ]:
def toy_lstm_kept(tokens_a, tokens_b, knob, threshold=50):
    """Returns (pct, verdict) -- pct can exceed 100% if the knob is so high the
    notebook keeps ADDING without ever erasing (it grows instead of fading); the
    verdict is always just "KEPT" or "NOT KEPT" against `threshold`, so nobody
    has to make sense of a number like 190%."""
    layer = SimpleLSTM(8, 8, seed=5)
    layer.b_f = layer.b_f + knob
    ha = hb = ca = cb = np.zeros(8)
    gaps = []
    for ta, tb in zip(tokens_a, tokens_b):
        xa, xb = live_mic.word_vector(ta), live_mic.word_vector(tb)
        ha, ca = layer.step(xa, ha, ca)
        hb, cb = layer.step(xb, hb, cb)
        gaps.append(np.abs(ca - cb).sum())
    ref = gaps[toy_memory.line_boundary_index(tokens_a)]
    pct = gaps[-1] / ref * 100
    verdict = "KEPT" if pct >= threshold else "NOT KEPT"
    if pct > 150:
        verdict += " (it grew instead of fading -- this knob adds without ever erasing)"
    return pct, verdict

short_pct_lstm, short_verdict_lstm = toy_lstm_kept(short_tokens, short_tokens_b, knob=3)
long_pct_lstm, long_verdict_lstm = toy_lstm_kept(long_tokens, long_tokens_b, knob=3)

plt.figure(figsize=(5, 3.5))
plt.bar(["short exchange", "long conversation"], [1 if short_pct_lstm >= 50 else 0, 1 if long_pct_lstm >= 50 else 0],
        color=["green" if short_pct_lstm >= 50 else "red", "green" if long_pct_lstm >= 50 else "red"])
plt.yticks([0, 1], ["NOT KEPT", "KEPT"])
plt.title("Knob +3: is line 1's fact still remembered?")
plt.tight_layout()
plt.show()
print(f"short exchange, knob +3: {short_verdict_lstm}")
print(f"long conversation, knob +3: {long_verdict_lstm}")

**Why:** both toy settings (RNN keep-rate 0.8, LSTM knob +3) are stated, untrained numbers, not
something the model learned -- changing the knob changes whether the fact survives, but on a long
enough conversation even a generous setting runs out of road (or, at the other extreme, starts
growing instead of fading -- still not the same as genuinely remembering). This is not a claim
that the trained LSTM beats a plain RNN; it is the same point either toy makes: **plain
sequential memory degrades over a long conversation.** Block 9 checks whether that is still true
for the TRAINED model.

### YOUR TURN: set the dial so the long conversation is remembered
Using the toy LSTM above, pick a `KNOB` so the long conversation's recording from 5c comes back
**KEPT**. Edit `KNOB` below and re-run -- the check tells you straight away. (Same toy, same
honest caveat: a stated setting, not something learned.)

In [ ]:
KNOB = 3   # YOUR TURN: try a bigger number if the check below says "not yet"

retained_pct, verdict = toy_lstm_kept(long_tokens, long_tokens_b, knob=KNOB)
print(f"knob {KNOB:+d}: {verdict}")
if "KEPT" in verdict and "NOT" not in verdict:
    print("check: OK -- you found a setting where the long conversation's first line survives.")
else:
    print("check: not yet -- try a bigger KNOB (more positive keeps more).")

<details><summary>Stuck? (click to expand)</summary>

- Still "NOT KEPT" at `KNOB = 3`? Try 5, then 8 -- the eraser needs to be told to keep much more
  than its default for a conversation this long.
- Once it flips to "KEPT", bigger numbers tend to stay "KEPT" too -- you don't need to find an
  exact edge, just clear the bar.
- This cell doesn't affect anything below it either way -- feel free to leave `KNOB` wherever you
  found it and move on.

</details>

## 5. Meeting minutes (Agenda block 9, 15 min)
Now put it together: your two-speaker transcript goes through the trained LSTM, which labels every turn as a
**decision**, an **action item** or plain **discussion**, and the minutes are written from the labels.

In [ ]:
from IPython.display import Markdown, display
from lstm_model import run_pipeline
from meeting_minutes import label_turns_model, generate_minutes

turns, output, turn_ends = run_pipeline(transcript=transcript)
labels = label_turns_model(turns, output, turn_ends)
for (speaker, text), label in zip(turns, labels):
    print(f"{label['category']:<12} {speaker}: {text}")
print()
display(Markdown(generate_minutes(turns, labels)))


**Look closely.** Did it find both decisions? (It misses decisions stated as plain facts, such as "Venue is confirmed", and
a bare "Sounds good." after a proposal. That is where a model trained on few examples is weakest.)


### Your own meeting (Agenda block 9, part)
Write 5 lines from a meeting of your own -- a CCA, project or club -- one turn per line,
`Name: what they said`. Labelling is optional, but if you add one, label **every** line with
`act` (action item), `dec` (decision) or `dis` (discussion) at the start.

<details><summary>More detail: accuracy, vocabulary, and the default example</summary>

The model was tested on 17 held-out meetings (clubs, projects, group work) and gets about
0.85-0.88 of turns right, but its vocabulary is small, so unusual words for your topic are read
as `<unk>` and ignored. Names show up as unseen words too, for the same reason -- the model
never actually reads them (they are dropped before it sees the text), so this is expected and
harmless.

The default below is the hook's own exchange, not an invented one -- so when you're asked which
earlier line the model should have remembered, there's a real answer.

</details>

In [ ]:
import os
import tempfile
from lstm_model import load_trained_model, label_probabilities
from pipeline import load_text, unknown_words
from IPython.display import Markdown, display

MY_MEETING = """
dis  Mei: Shall we move it to Friday?
dec  Ravi: Sure.
act  Mei: Can you book the room?
act  Ravi: Sure.
dis  Mei: Great, thanks.
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as f:
    f.write(MY_MEETING)
    my_meeting_path = f.name
try:
    my_text, my_gold = load_text(my_meeting_path)
except ValueError as e:
    # Strip the temp-file path off the front of the message -- a participant doesn't need to
    # see where this notebook happened to save their typed text, just what to fix.
    message = str(e).replace(my_meeting_path, "").strip(" ,:")
    print("Couldn't read that meeting:", message)
    print("Hint: label every line or none of them, and write each line as 'Name: text'")
    print("(something after the colon).")
    my_text = None
finally:
    os.remove(my_meeting_path)

if my_text is not None:
    my_turns, my_output, my_ends = run_pipeline(transcript=my_text)
    my_labels = label_turns_model(my_turns, my_output, my_ends)
    my_probs = label_probabilities(my_turns, trained)

    for i, ((speaker, text), label, probs) in enumerate(zip(my_turns, my_labels, my_probs)):
        model_cat = label["category"]
        confidence = probs[trained["labels"].index(model_cat)]
        if my_gold:
            your_cat = my_gold[i]
            mark = "" if your_cat == model_cat else "  <- X"
            print(f"{your_cat:<12} (you)   {model_cat:<12} (model, {confidence:.0%} sure)   {speaker}: {text}{mark}")
        else:
            print(f"{model_cat:<12} (model, {confidence:.0%} sure)   {speaker}: {text}")
    if not my_gold:
        print()
        print("No labels given -- add act/dec/dis at the start of every line above and run again")
        print("to see where you and the model agree.")

    print()
    display(Markdown(generate_minutes(my_turns, my_labels)))

    unknown = unknown_words(my_turns)
    print()
    if unknown:
        print(f"Words the model never saw in training (read as <unk>): {', '.join(unknown)}")
    vocab_size = len(load_trained_model()["vocab"])
    print(f"The model knows {vocab_size} words; anything else is read as <unk>.")

    print()
    print("Pick one line it got wrong. Which EARLIER line should it have remembered?")
    print("(On the default example: the first \"Sure.\" is nearly 50-50 -- it needs the line")
    print("before it, \"Shall we move it to Friday?\", to be read as agreeing to a plan.)")


**Add a line before it, and watch it change.** Try it on the same knife-edge "Sure." by
putting one more line in front of the whole meeting:


In [ ]:
if my_text is not None:
    with_context = "Mei: Let's plan the showcase." + chr(10) + my_text
    with_context_turns, with_context_output, with_context_ends = run_pipeline(transcript=with_context)
    with_context_probs = label_probabilities(with_context_turns, trained)

    sure_count = 0
    sure_labels = ["first \"Sure.\" (after the proposal)", "second \"Sure.\" (after the request)"]
    for (speaker, text), probs in zip(with_context_turns, with_context_probs):
        if text == "Sure.":
            top = max(zip(trained["labels"], probs), key=lambda lp: lp[1])
            which = sure_labels[sure_count] if sure_count < len(sure_labels) else f"Sure. #{sure_count + 1}"
            print(f"{speaker}: {which:32s} -> {top[0]} ({top[1]:.0%})")
            sure_count += 1


One extra line of context earlier in the meeting is enough to flip the model's answer on
that first "Sure." -- from the default's near-50-50 miss to a (still narrow) correct
`decision`, 56% against 43%. **Caveat:** this is two examples on one knife-edge line, not a
general claim that "more context always helps" -- the model can just as easily be nudged the
wrong way by an earlier line, and most turns in this notebook aren't this close to begin with.


### 9b. Talk your own meeting (live mic) *(2-3 minutes, solo or in pairs)*
**Predict:** will the minutes generated from a SPOKEN meeting look anything like the ones you'd
write by hand?

**Try it:** talk through a short meeting, 4-6 lines, alone (playing both people, pausing ~2s
between lines) or with a partner -- include a decision, an action item, and some plain
discussion. You have up to 35s -- no need to rush.

In [ ]:
MEETING_LIVE, meeting_path, _ = live_mic.say("block9_meeting", model, seconds=35, minimum_words=6)
print("heard (flat):", MEETING_LIVE)

meeting_result = model.transcribe(meeting_path, word_timestamps=True, language="en")
meeting_words_ts = [{"start": w["start"], "end": w["end"], "text": w["word"]}
                     for seg in meeting_result["segments"] for w in seg.get("words", [])]
live_meeting_text = segments_to_transcript(meeting_words_ts, pause_gap=1.2)
print(live_meeting_text)

### YOUR TURN: label your meeting before the model does
Looking at the turns printed above, write one label per turn: `act` (action item), `dec`
(decision), or `dis` (discussion/plain talk) -- same order as the turns above.

In [ ]:
_live_meeting_turn_count = len([l for l in live_meeting_text.strip().split(chr(10)) if l])
MY_LABELS = ["dis"] * _live_meeting_turn_count   # YOUR TURN: your own act/dec/dis guesses, same order as above

if len(MY_LABELS) != _live_meeting_turn_count:
    print(f"check: not yet -- {len(MY_LABELS)} labels for {_live_meeting_turn_count} turns; one label per turn.")
    MY_LABELS_OK = False
elif any(l not in ("act", "dec", "dis") for l in MY_LABELS):
    print("check: not yet -- each label must be 'act', 'dec' or 'dis'.")
    MY_LABELS_OK = False
else:
    print("check: OK -- compared against the model's own labels below.")
    MY_LABELS_OK = True

<details><summary>Stuck? (click to expand)</summary>

- Wrong number of labels -- count the turns printed above and match that length exactly.
- A typo in a label -- only `act`, `dec` or `dis`, lowercase.
- Skipping this is fine -- the model's labels print either way, just without your own guesses
  alongside them.

</details>

In [ ]:
live_meeting_turns, live_meeting_output, live_meeting_ends = run_pipeline(transcript=live_meeting_text)
live_meeting_labels = label_turns_model(live_meeting_turns, live_meeting_output, live_meeting_ends)
LABEL_MAP = {"act": "action_item", "dec": "decision", "dis": "discussion"}
for i, ((speaker, text), label) in enumerate(zip(live_meeting_turns, live_meeting_labels)):
    model_cat = label["category"]
    if MY_LABELS_OK:
        your_cat = LABEL_MAP[MY_LABELS[i]]
        mark = "" if your_cat == model_cat else "  <- different"
        print(f"{your_cat:<12} (you)   {model_cat:<12} (model)   {speaker}: {text}{mark}")
    else:
        print(f"{model_cat:<12} {speaker}: {text}")
print()
display(Markdown(generate_minutes(live_meeting_turns, live_meeting_labels)))

unknown9 = unknown_words(live_meeting_turns)
if unknown9:
    print()
    print(f"Words the model never saw in training (read as <unk>): {', '.join(unknown9)}")

**Why:** same pipeline as the typed cell above (`run_pipeline`, `label_turns_model`,
`generate_minutes`) -- the only difference is the transcript came from your own voice through
Whisper and the same pause-based speaker splitter blocks 1-2 used, instead of being typed in. The
typed cell above stays as the fallback and as a labelled check (this spoken version has no gold
labels to compare against -- there's no easy way to speak "dec"/"act"/"dis" -- so there's no
right/wrong mark here, just what the model made of it). A pause that lands mid-sentence, or two
people talking over each other, can still split or merge turns wrong, same known limitation as the
hook's own backup-WAV pacing notes in `demo/FACILITATOR.md`.

**See it:** does the trained model's last answer change more when you edit the FIRST line of
your meeting above, or the line right before the last one?

In [ ]:
if len(live_meeting_turns) >= 6:
    reach_turns = live_meeting_turns
else:
    # A short meeting (under 6 turns) doesn't show this reliably -- the effect needs some
    # distance between "line 1" and "the line before last" to be a fair test either way.
    print(f"Your spoken meeting (9b) only had {len(live_meeting_turns)} turns -- using the longer sample meeting from blocks 1-2 instead, so the two changes are actually comparable.")
    reach_turns = turns   # the block 1-2 sample meeting, always 9 turns

ALT_LINE_A = "The weather has been unusually warm this week."
ALT_LINE_B = "The printer on the third floor needs new toner."

def reach_test(all_turns, replace_idx, replacement_text):
    modified = list(all_turns)
    speaker, _ = modified[replace_idx]
    modified[replace_idx] = (speaker, replacement_text)
    return label_probabilities(modified, trained)[-1]

baseline_last = label_probabilities(reach_turns, trained)[-1]
changed_first = reach_test(reach_turns, 0, ALT_LINE_A)
changed_before_last = reach_test(reach_turns, len(reach_turns) - 2, ALT_LINE_B)

shift_first = float(np.abs(changed_first - baseline_last).sum() / 2)      # total variation distance, 0-1
shift_before_last = float(np.abs(changed_before_last - baseline_last).sum() / 2)

plt.figure(figsize=(5, 3.5))
bars = plt.bar(["change line 1", "change the line\nbefore last"],
               [shift_first * 100, shift_before_last * 100], color=["steelblue", "darkorange"])
plt.ylabel("how much the LAST line's answer moved")
plt.title("The trained model's real memory reach")
plt.ylim(0, 100)
for bar, v in zip(bars, [shift_first * 100, shift_before_last * 100]):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 2, f"{v:.0f}%", ha="center")
plt.tight_layout()
plt.show()
print(f"changing line 1:              last line's answer moved {shift_first:.0%}")
print(f"changing the line before last: last line's answer moved {shift_before_last:.0%}")
print("(checked against several different replacement lines: on this model, either change can")
print(" move the last answer by a similar, modest amount -- see the Why below.)")

**Why:** `label_probabilities` is the real trained ensemble (the same one used everywhere else
in this notebook), not a toy. Checked against 6 different replacement lines at each position (not
just the one shown): changing line 1 and changing the line before last move the last line's
answer by a similar, modest amount on average (around 2-3%, neither consistently "barely" nor
consistently "a lot") -- the trained model's real dependence on context is messier than the toy
RNN's clean, position-based fading. **What this does NOT show:** a strict "only the previous
turn matters" rule -- that was the toy's story, not a proven property of the trained model. What
it DOES show: a small, shallow model like this one doesn't have a clean, predictable memory
either way -- which is itself the honest point. Never read this as "the trained LSTM remembers
as little as a 1-layer toy" -- it is a genuinely better model at the one-turn scale -- but its
context-sensitivity is not a simple recency law.

## 6. Wrap-up (Agenda block 10, 5 min)
**The whole journey, in one sentence:** sound becomes words, memory turns words into meaning,
and meaning becomes minutes.

**Three things this model honestly still gets wrong, seen in this notebook:**

1. A decision stated as a plain fact ("Venue is confirmed...") -- no "agree" or "let's" to catch
   it on.
2. Words it never saw in training -- read as `<unk>` and ignored.
3. A bare agreement to a proposal ("Sure." after "Shall we move it to Friday?") -- the model
   reads context the same way the room does, but doesn't always land on the right answer from it.

Take-home challenge: add your own 5 lines from "your own meeting" (section 5) and see whether a
retrained model would learn them -- the full code, including training, is with the facilitators
(not in this zip).


### Going further
* Change `MODEL` in section 1 to `small.en` or `medium.en`. How much does the transcript improve, and how much slower is it?
* Record a real two-person conversation (pause between speakers) and run it through `get_transcript`.
* Section 2 again with your own noise level or a different recording.

### YOUR TURN: explain it in your own words
In 1-2 sentences: why does a long conversation's memory fade so much more than a short one's?
Edit the string below.

In [ ]:
MY_WHY = """YOUR TURN: write your own 1-2 sentence explanation here."""
print(MY_WHY)

### What I built today

In [ ]:
print("What I built today:")

_recorded_vars = ["SAID_LIVE", "NOISY_LIVE", "WORD_LIVE", "SENTENCE_LIVE", "CATS_LIVE", "DOGS_LIVE",
                   "SHORT_A", "SHORT_B", "LONG_A", "LONG_B", "GATE_LIVE", "REQUEST_LIVE", "PROPOSAL_LIVE",
                   "FACT_LIVE", "SURE_REQ_LIVE", "SURE_PROP_LIVE", "EXCHANGE_LIVE", "MEETING_LIVE",
                   "SHORT_CONV", "LONG_CONV"]
_recorded_count = sum(1 for v in _recorded_vars if v in globals())
print(f"- {_recorded_count}/{len(_recorded_vars)} live-mic prompts recorded (or used the fallback for)")

if "KNOB" in globals():
    print(f"- the memory dial you set: KNOB = {KNOB!r}")
if "MY_SENTENCE_A" in globals() and "MY_SENTENCE_B" in globals():
    print(f"- your own first-word pair: {MY_SENTENCE_A!r} / {MY_SENTENCE_B!r}")
if "MY_LABELS" in globals():
    print(f"- your own meeting's labels: {MY_LABELS!r}")
if "MY_WHY" in globals():
    print(f"- your own explanation: {MY_WHY.strip()[:120]!r}")